# Session 4 Homework · Solutions (teacher copy)

The Module 1 capstone, worked end-to-end with teaching notes. All code runs top-to-bottom.

**Grading toward the success criterion** (concrete artifact — named metrics begin Session 8):

- **data:** prepared table reports `0` missing values and `0` text columns
- **model:** data is split and the model is fit to predict `price_lakhs`
- **evaluation:** a test `.score()` (R²) is printed
- **insight:** ≥ 2 sentences naming a feature and the direction it pushes price, with a note of caution

This homework rehearses exactly what the **Module 1 checkpoint** assesses. Accept reasonable variants: mean vs median fill, `drop_first=True` in encoding, a different `random_state`, and any sensible insight (area↑→price↑, city_center↑→price↑, distance↑→price↓, age↑→price↓).

## Step 1 · DATA — clean, then encode

In [ ]:
import pandas as pd

homes = pd.read_csv("../../../datasets/secondary/housing.csv")

homes["age_years"] = homes["age_years"].fillna(homes["age_years"].median())
homes["distance_to_center_km"] = homes["distance_to_center_km"].fillna(homes["distance_to_center_km"].median())

homes = pd.get_dummies(homes, columns=["neighborhood_type"], dtype=int)

print("missing values:", homes.isna().sum().sum())
print("text columns:  ", len(homes.select_dtypes(include="object").columns))
homes.head()

**Expected:** both checks print `0`. The table now has the three `neighborhood_type_*` columns in place of the text column. A student who skipped either fix will fail a later step loudly (a `ValueError` on `fit`, or a leftover `NaN`) — that's a useful diagnostic, not a disaster.

## Step 2 · MODEL — split, then fit

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

y = homes["price_lakhs"]
X = homes.drop(columns=["price_lakhs"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

model = LinearRegression()
model.fit(X_train, y_train)
print("model trained on", X_train.shape[0], "homes.")

## Step 3 · EVALUATION — score on held-out data

In [ ]:
print("training score:", round(model.score(X_train, y_train), 3))
print("test score:    ", round(model.score(X_test, y_test), 3))

**Expected:** both scores are high (≈ 0.9) and close together, because price was built as a mostly-linear function of the features plus noise. Close scores → the model generalised (learned), rather than memorizing the training homes. A much lower test score is the red flag we study in Session 16 — but it shouldn't appear here.

## Step 4 · INSIGHT — change one thing

In [ ]:
scenario = X_test.iloc[[0]].copy()
base_prediction = model.predict(scenario)[0]

bigger = scenario.copy()
bigger["area_sqft"] = bigger["area_sqft"] + 500
bigger_prediction = model.predict(bigger)[0]

print(f"base predicted price:        {base_prediction:6.1f} lakhs")
print(f"after +500 sqft:             {bigger_prediction:6.1f} lakhs   (change: {bigger_prediction - base_prediction:+.1f})")

**Model insight (accept any well-reasoned variant):**

"In this data, a larger built-up area is associated with a higher predicted price — adding 500 square feet raised the predicted price by a clear margin. I'd be cautious calling this a guaranteed rule, because the model only found a pattern in these particular listings (prices in lakhs, one city's market) and it doesn't know about anything outside its features — location quality, build condition, or the year of sale."

*Grading:* require ≥ 2 sentences, a named feature with a direction, and a note of caution (association not causation, or limits of the data). Other strong choices: switching the one-hot columns to `city_center` raises the price; more distance from the center lowers it.

## One last reflection

**Teaching note:** there is no right answer here — it's a metacognition prompt. Common honest responses: *data* felt hardest (remembering to both fill *and* encode before anything else works), or *insight* (turning a number into a careful sentence). Use the answer to decide what to reinforce in the Module 1 checkpoint review.